# 04 · Spin-Orbital MP2

这一节把上一节的 spatial orbital MP2 改写成 spin orbital 形式。

核心想法很简单：

1. 每个空间轨道复制成 alpha 和 beta 两个 spin orbital
2. 一电子量用 `np.kron(I_spin, spatial_matrix)` 扩展
3. 双电子积分用一个 spin selection tensor 和空间积分做 `np.kron`
4. 用 spin-orbital MP2 公式计算相关能

如果 alpha 和 beta 使用不同的空间轨道，这套写法就是 UMP2 的自然语言；本 notebook 先从 RHF 的同一套空间轨道复制 alpha/beta，因此数值会与闭壳层 RMP2 完全一致。

---

## 1 · Spin-Orbital MP2 公式

在 spin orbital 中，每个轨道指标都已经包含了自旋：

$$
p \equiv (\phi_p, \sigma_p), \quad \sigma_p \in \{\alpha, \beta\}
$$

MP2 相关能写成：

$$
\boxed{
E_{\text{MP2}}^{(2)} = \frac{1}{4}
\sum_{ij}^{\text{occ}} \sum_{ab}^{\text{vir}}
\frac{|\langle ij \Vert ab \rangle|^2}
{\varepsilon_i + \varepsilon_j - \varepsilon_a - \varepsilon_b}
}
$$

在 chemist notation 下：

$$
\langle ij \Vert ab \rangle = (ia|jb) - (ib|ja)
$$

这里的 $i,j,a,b$ 都是 spin orbital 指标。和 spatial-orbital 公式相比，自旋求和没有提前做掉，所以公式变成了「反对称化积分平方」并带有 $1/4$。

---

## 2 · RHF 参考态与空间轨道积分

和 spatial-orbital MP2 一样，这里直接用 PySCF `ao2mo` 生成 s1 symmetry 的完整空间 MO 积分。
`einsum` 只作为短小的 sanity check：它适合方法验证，但不是高性能积分变换的实现方式。

In [1]:
import numpy as np
from pyscf import gto, scf, mp, ao2mo

# -------- H2O / STO-3G --------
mol = gto.M(atom="O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587", basis="sto-3g")

mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()

C = mf.mo_coeff
mo_energy = mf.mo_energy
nmo = C.shape[1]
nelec = mol.nelectron
nocc_spatial = nelec // 2
nvir_spatial = nmo - nocc_spatial

# Spatial MO integrals, eri_mo[p,q,r,s] = (pq|rs)
eri_mo = ao2mo.kernel(mol, C, aosym="s1").reshape(nmo, nmo, nmo, nmo)

# Sanity check against the explicit four-index transform.
eri_ao = mol.intor("int2e_sph")
eri_mo_einsum = np.einsum("up,vq,kr,ls,uvkl->pqrs", C, C, C, C, eri_ao, optimize=True)
ao2mo_err = np.max(np.abs(eri_mo - eri_mo_einsum))

print(f"RHF energy = {mf.e_tot:.10f} Hartree")
print(f"spatial nmo = {nmo}, occupied spatial orbitals = {nocc_spatial}")
print(f"eri_mo shape = {eri_mo.shape}")
print(f"max|ao2mo - einsum| = {ao2mo_err:.2e}")
assert ao2mo_err < 1e-10

RHF energy = -74.9630631297 Hartree
spatial nmo = 7, occupied spatial orbitals = 5
eri_mo shape = (7, 7, 7, 7)


---

## 3 · 用 `kron` 构造 spin orbital

采用最直观的排序：

```text
0 ... nmo-1          : alpha spatial orbitals
nmo ... 2*nmo-1      : beta  spatial orbitals
```

于是轨道能量就是复制两份：

$$
\varepsilon^{\text{SO}} = (\varepsilon^\alpha, \varepsilon^\beta)
$$

MO 系数和一电子矩阵都可以写成 block diagonal 形式：

$$
C^{\text{SO}} = I_{2\times2} \otimes C^{\text{spatial}}
$$

$$
h^{\text{SO}} = I_{2\times2} \otimes h^{\text{spatial}}
$$

双电子积分的自旋选择规则来自自旋积分：

$$
(p\sigma, q\tau | r\upsilon, s\omega)
= \delta_{\sigma\tau}\delta_{\upsilon\omega}(pq|rs)
$$

也就是一个 `(2,2,2,2)` 的 spin tensor 与空间积分做 Kronecker product。

In [2]:
spin_eye = np.eye(2)
spin_factor = np.einsum("st,uv->stuv", spin_eye, spin_eye)

# AO spin-orbital -> MO spin-orbital coefficient matrix.
# Ordering: all alpha AOs/MOs first, then all beta AOs/MOs.
C_spin = np.kron(spin_eye, C)

# One-electron quantities: two identical spin blocks.
eps_spin = np.kron(np.ones(2), mo_energy)

# Two-electron spin-orbital integrals.
# eri_spin[P,Q,R,S] = (PQ|RS), where P,Q,R,S are spin-orbital indices.
eri_spin = np.kron(spin_factor, eri_mo)

nspin = 2 * nmo
print(f"C_spin shape   = {C_spin.shape}")
print(f"eps_spin shape = {eps_spin.shape}")
print(f"eri_spin shape = {eri_spin.shape}")
print(f"spin_factor nonzero elements = {np.count_nonzero(spin_factor)} / {spin_factor.size}")

C_spin shape   = (14, 14)
eps_spin shape = (14,)
eri_spin shape = (14, 14, 14, 14)
spin_factor nonzero elements = 4 / 16


---

## 4 · 占据与虚 spin orbitals

RHF 参考态中，前 `nocc_spatial` 个 alpha 轨道占据，前 `nocc_spatial` 个 beta 轨道也占据。

用当前排序，occupied / virtual spin orbitals 为：

```text
occ = alpha_occ + beta_occ
vir = alpha_vir + beta_vir
```

In [3]:
alpha_occ = np.arange(0, nocc_spatial)
alpha_vir = np.arange(nocc_spatial, nmo)

beta_occ = nmo + np.arange(0, nocc_spatial)
beta_vir = nmo + np.arange(nocc_spatial, nmo)

occ = np.r_[alpha_occ, beta_occ]
vir = np.r_[alpha_vir, beta_vir]

print(f"occupied spin orbitals = {occ}")
print(f"virtual  spin orbitals = {vir}")
print(f"nocc spin = {len(occ)}, nvir spin = {len(vir)}")

occupied spin orbitals = [ 0  1  2  3  4  7  8  9 10 11]
virtual  spin orbitals = [ 5  6 12 13]
nocc spin = 10, nvir spin = 4


---

## 5 · 四重循环实现

这次公式里的每个指标都是 spin orbital。

反对称化积分：

$$
\langle ij \Vert ab \rangle = (ia|jb) - (ib|ja)
$$

能量：

$$
E^{(2)} = \frac{1}{4}\sum_{ijab}\frac{\langle ij\Vert ab\rangle^2}{\Delta_{ij}^{ab}}
$$

In [4]:
def spin_mp2_energy_loop(eps_spin, eri_spin, occ, vir):
    emp2 = 0.0

    for i in occ:
        for j in occ:
            for a in vir:
                for b in vir:
                    gijab = eri_spin[i, a, j, b] - eri_spin[i, b, j, a]
                    denom = eps_spin[i] + eps_spin[j] - eps_spin[a] - eps_spin[b]
                    emp2 += 0.25 * gijab * gijab / denom

    return emp2

emp2_spin_loop = spin_mp2_energy_loop(eps_spin, eri_spin, occ, vir)
print(f"Spin-orbital MP2 correlation (loop) = {emp2_spin_loop:.10f} Hartree")
print(f"MP2 total energy                    = {mf.e_tot + emp2_spin_loop:.10f} Hartree")

Spin-orbital MP2 correlation (loop) = -0.0355668182 Hartree
MP2 total energy                    = -74.9986299479 Hartree


---

## 6 · `einsum` 向量化实现

切出 spin-orbital 的 `(ia|jb)` 张量：

```python
iajb[I,A,J,B] = (I A | J B)
```

然后交换 `A` 和 `B` 得到 `(ib|ja)`。

In [5]:
def spin_mp2_energy_einsum(eps_spin, eri_spin, occ, vir):
    iajb = eri_spin[np.ix_(occ, vir, occ, vir)]
    ibja = iajb.transpose(0, 3, 2, 1)
    gijab = iajb - ibja

    eps_occ = eps_spin[occ]
    eps_vir = eps_spin[vir]
    denom = (
        eps_occ[:, None, None, None]
        - eps_vir[None, :, None, None]
        + eps_occ[None, None, :, None]
        - eps_vir[None, None, None, :]
    )

    emp2 = 0.25 * np.einsum("iajb,iajb,iajb->", gijab, gijab, 1.0 / denom, optimize=True)
    return emp2

emp2_spin_einsum = spin_mp2_energy_einsum(eps_spin, eri_spin, occ, vir)

print(f"Spin-orbital MP2 correlation (einsum) = {emp2_spin_einsum:.10f} Hartree")
print(f"loop - einsum difference              = {abs(emp2_spin_loop - emp2_spin_einsum):.2e} Hartree")

Spin-orbital MP2 correlation (einsum) = -0.0355668182 Hartree
loop - einsum difference              = 2.08e-17 Hartree


---

## 7 · 回到 spatial-orbital 公式检查

同一个 RHF 参考态下，spin-orbital MP2 应该和上一节的闭壳层 spatial-orbital MP2 给出相同相关能。

In [6]:
def spatial_mp2_energy(mo_energy, eri_mo, nocc):
    eps_occ = mo_energy[:nocc]
    eps_vir = mo_energy[nocc:]

    iajb = eri_mo[:nocc, nocc:, :nocc, nocc:]
    ibja = iajb.transpose(0, 3, 2, 1)

    denom = (
        eps_occ[:, None, None, None]
        - eps_vir[None, :, None, None]
        + eps_occ[None, None, :, None]
        - eps_vir[None, None, None, :]
    )

    return np.einsum("iajb,iajb,iajb->", iajb, 2.0 * iajb - ibja, 1.0 / denom, optimize=True)

emp2_spatial = spatial_mp2_energy(mo_energy, eri_mo, nocc_spatial)

print(f"Spatial-orbital MP2 correlation = {emp2_spatial:.10f} Hartree")
print(f"Spin-orbital MP2 correlation    = {emp2_spin_einsum:.10f} Hartree")
print(f"Difference                      = {abs(emp2_spatial - emp2_spin_einsum):.2e} Hartree")

Spatial-orbital MP2 correlation = -0.0355668182 Hartree
Spin-orbital MP2 correlation    = -0.0355668182 Hartree
Difference                      = 6.94e-18 Hartree


---

## 8 · 与 PySCF MP2 对照

In [7]:
mp2_ref = mp.MP2(mf)
mp2_ref.verbose = 0
emp2_ref, t2_ref = mp2_ref.kernel()

print(f"Our spin-orbital MP2:  {emp2_spin_einsum:.10f} Hartree")
print(f"PySCF MP2 correlation: {emp2_ref:.10f} Hartree")
print(f"Difference:            {abs(emp2_spin_einsum - emp2_ref):.2e} Hartree")

print(f"\nOur MP2 total energy:   {mf.e_tot + emp2_spin_einsum:.10f} Hartree")
print(f"PySCF MP2 total energy: {mp2_ref.e_tot:.10f} Hartree")

Our spin-orbital MP2:  -0.0355668182 Hartree
PySCF MP2 correlation: -0.0355668182 Hartree
Difference:            6.94e-18 Hartree

Our MP2 total energy:   -74.9986299479 Hartree
PySCF MP2 total energy: -74.9986299479 Hartree


---

## 9 · 总结

| 模块 | 核心内容 |
|------|----------|
| **spin orbital 排序** | alpha block followed by beta block |
| **轨道能量** | `eps_spin = np.kron(np.ones(2), mo_energy)` |
| **自旋选择规则** | $\delta_{\sigma_p\sigma_q}\delta_{\sigma_r\sigma_s}$ |
| **空间 MO 积分** | `ao2mo.kernel(mol, C, aosym="s1")`；`einsum` 只做验证 |
| **双电子积分** | `eri_spin = np.kron(spin_factor, eri_mo)` |
| **反对称化积分** | $\langle ij\Vert ab\rangle=(ia|jb)-(ib|ja)$ |
| **spin-orbital MP2** | $\frac14\sum_{ijab}\langle ij\Vert ab\rangle^2/\Delta_{ij}^{ab}$ |
| **闭壳层检查** | 与 spatial-orbital MP2 完全一致 |

从这里往 UMP2 推广时，主要变化是 alpha/beta 使用不同的 MO 系数和轨道能量；概念上仍然是在 spin orbital 空间里做同一件事。